# TFM — Anonimización y Privacidad en IA Sanitaria

Walkthrough en Colab del pipeline experimental completo. Este notebook reproduce
las **once fases** del Capítulo 3 sobre el conjunto Diabetes 130-US hospitals
(UCI Machine Learning Repository) y es **equivalente** a ejecutar `scripts/01_*.py`
a `scripts/11_*.py` en orden — todas las funciones críticas se importan de `src/`
para garantizar identidad byte a byte con los scripts.

## Fases

1. Configuración del entorno e imports
2. Ingesta y limpieza del conjunto Diabetes 130-US
3. Perfilado de privacidad (riesgo basal de reidentificación)
4. Reducción de dimensionalidad (agrupación CIE-9)
5. Partición estratificada y modelos baseline
6. Exportación para ARX Desktop (k-anonimidad)
7. Evaluación post-ARX (utilidad y equidad)
8. Aplicación de Privacidad Diferencial (n=20 réplicas)
9. Auditoría adversarial MIA Black-Box
10. Tests de significancia estadística con Bonferroni (McNemar y Wilcoxon)
11. Defensa en profundidad: combinación k=10 + DP
12. Extensión sintáctica: l-diversidad y t-closeness
13. MIA sobre configuraciones estrictas l/t
14. McNemar Bonferroni sobre 4 configs extremas l/t
15. Artefactos generados

## 1. Configuración del entorno

In [ ]:
# Dependencias (Colab no las trae todas por defecto)
!pip install -q "scikit-learn==1.6.1" "diffprivlib==0.6.6" "adversarial-robustness-toolbox==1.20.1" "ucimlrepo==0.0.7"

# Si el notebook se ejecuta sin acceso a src/ (ej. Colab limpio), clonamos el repo.
import os, sys, shutil, subprocess

REPO_URL = "https://github.com/alexherrland/tfm-anonimizacion-ia.git"
CLONE_DIR = "_repo"

def _find_repo_root(start: str) -> str:
    """Devuelve el primer directorio bajo `start` que contenga src/ y scripts/."""
    for root, dirs, _ in os.walk(start):
        if "src" in dirs and "scripts" in dirs:
            return root
    return start

if not (os.path.isdir("src") or os.path.isdir("../src")):
    if os.path.isdir(CLONE_DIR):
        print(f"  → Reaprovechando {CLONE_DIR}/ (existe de una ejecución previa)")
    else:
        print(f"  → src/ no encontrado, clonando {REPO_URL} ...")
        subprocess.run(["git", "clone", "-q", REPO_URL, CLONE_DIR], check=True)
    repo_root = _find_repo_root(CLONE_DIR)
    os.chdir(repo_root)
    print(f"  → cd {os.getcwd()}")

# Asegurar que src/ es importable desde el cwd o desde la carpeta padre
for _p in (".", ".."):
    if os.path.isdir(os.path.join(_p, "src")) and _p not in sys.path:
        sys.path.insert(0, _p)
print("cwd:", os.getcwd())
print("src/ accesible:", any(os.path.isdir(os.path.join(p, "src")) for p in sys.path))

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import wilcoxon, chi2

import diffprivlib.models as dpm
from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.estimators.classification.scikitlearn import ScikitlearnLogisticRegression
from ucimlrepo import fetch_ucirepo
# Imports del repositorio (src/) — garantizan identidad con los scripts 01–11
from src import config as _cfg
from src.data_loader import load_clean_reduced
from src.preprocessing import (
    binarize_target, joint_ohe, fit_scaler, stratified_split,
    percentile_data_norm, percentile_bounds,
)
from src.models import build_baseline_models, build_dp_models
from src.kanon import evaluate_kanon as _evaluate_kanon_src
from src.ldiv_tclos import (
    evaluate_ldiv_tclos, fairness_ldiv_tclos, verify_constraints,
)
from src.differential_privacy import sweep_dp, dp_on_kanonimized
from src.statistical_tests import (
    apply_bonferroni, mcnemar_test, build_mcnemar_table,
    build_wilcoxon_table, wilcoxon_one_sample,
)


warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
K_VALUES = [2, 5, 10, 25, 50]
EPSILON_VALUES = [0.1, 0.5, 1.0, 5.0, 10.0]
N_REPETITIONS_DP = 20

DP_SEEDS = [42, 137, 271, 314, 1729, 2718, 3141, 6022, 8128, 9999, 10007, 11113, 13121, 17171, 19273, 23131, 29327, 31193, 33391, 37397]

# Rutas alineadas con la estructura canónica de arx_kit/.
# Al ejecutarse en Colab/local crea la jerarquía completa en el cwd.
WORKDIR = Path("arx_kit")
INPUTS_DIR = WORKDIR / "inputs"
HIERARCHIES_DIR = INPUTS_DIR / "arx_hierarchies"
ARX_OUTPUTS_DIR = WORKDIR / "arx_outputs"
RESULTS_KANON_DIR = WORKDIR / "results" / "kanon"
RESULTS_DP_DIR = WORKDIR / "results" / "dp"
RESULTS_LDIV_TCLOS_DIR = WORKDIR / "results" / "ldiv_tclos"
RESULTS_MIA_DIR = WORKDIR / "results" / "mia"
FIGURES_DIR = Path("results")  # PNGs de la memoria
for _d in (INPUTS_DIR, HIERARCHIES_DIR, ARX_OUTPUTS_DIR,
           RESULTS_KANON_DIR, RESULTS_DP_DIR,
           RESULTS_LDIV_TCLOS_DIR, RESULTS_MIA_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

## 2. Ingesta y limpieza del conjunto Diabetes 130-US

El conjunto de datos contiene 101.766 registros de encuentros hospitalarios
de pacientes diagnosticados con diabetes. Se aplican tres estrategias de
preprocesamiento para gestionar los valores ausentes: supresión de la
variable `weight` (96,85 % de nulos), imputación semántica de las pruebas
clínicas no realizadas y eliminación de filas con nulos residuales en
variables troncales.

In [ ]:
dataset = fetch_ucirepo(id=296)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.replace("?", np.nan, inplace=True)

df.drop(columns=["weight"], inplace=True)
df[["max_glu_serum", "A1Cresult", "medical_specialty", "payer_code"]] = (
    df[["max_glu_serum", "A1Cresult", "medical_specialty", "payer_code"]].fillna("No_Registrado")
)
df.dropna(subset=["race", "diag_1", "diag_2", "diag_3"], inplace=True)
print(f"Dimensiones tras limpieza: {df.shape}")

## 3. Perfilado de privacidad: riesgo basal de reidentificación

Se identifican cinco cuasi-identificadores (QIDs) consistentes con un
modelo de atacante realista con acceso a registros administrativos. El
riesgo basal se cuantifica mediante el tamaño mínimo de las clases de
equivalencia: un valor $k=1$ indica vulnerabilidad extrema a ataques de
enlace.

In [ ]:
qids = ["race", "gender", "age", "time_in_hospital", "admission_type_id"]
equivalence_classes = df.groupby(qids).size().reset_index(name="size")

n_total = len(df)
n_unique = equivalence_classes[equivalence_classes["size"] == 1]["size"].sum()
n_vulnerable_k5 = equivalence_classes[equivalence_classes["size"] < 5]["size"].sum()

print(f"Combinaciones únicas de QIDs: {len(equivalence_classes)}")
print(f"Pacientes con k=1 (vulnerabilidad extrema): {n_unique} ({n_unique/n_total*100:.2f} %)")
print(f"Pacientes con k<5: {n_vulnerable_k5} ({n_vulnerable_k5/n_total*100:.2f} %)")

## 4. Reducción de dimensionalidad

La cardinalidad original de los códigos CIE-9 produce una matriz One-Hot
Encoding de 2.405 dimensiones, incompatible con la inyección de ruido en
Privacidad Diferencial. Se agrupan los códigos en nueve macrocategorías
médicas mediante una función de mapeo basada en el conocimiento del
dominio, reduciendo el espacio a 121 dimensiones.

In [ ]:
def map_diagnosis(code):
    if pd.isna(code) or code == "?":
        return "Other"
    if isinstance(code, str) and (code.startswith("V") or code.startswith("E")):
        return "Other"
    try:
        v = float(code)
    except (TypeError, ValueError):
        return "Other"
    if 250 <= v < 251:                  return "Diabetes"
    if 390 <= v <= 459 or v == 785:     return "Circulatory"
    if 460 <= v <= 519 or v == 786:     return "Respiratory"
    if 520 <= v <= 579 or v == 787:     return "Digestive"
    if 580 <= v <= 629 or v == 788:     return "Genitourinary"
    if 710 <= v <= 739:                 return "Musculoskeletal"
    if 800 <= v <= 999:                 return "Injury"
    if 140 <= v <= 239:                 return "Neoplasms"
    return "Other"

for column in ("diag_1", "diag_2", "diag_3"):
    df[f"{column}_category"] = df[column].apply(map_diagnosis)

df_reduced = df.drop(columns=["diag_1", "diag_2", "diag_3", "medical_specialty"])
X_raw = df_reduced.drop(columns=["readmitted"])
y_bin = (df_reduced["readmitted"] != "NO").astype(int)
X_encoded = pd.get_dummies(X_raw, drop_first=True)
print(f"Dimensiones tras reducción semántica: {X_encoded.shape}")

## 5. Partición estratificada y modelos baseline

El conjunto se divide en 80 % entrenamiento y 20 % prueba con muestreo
estratificado. Se entrenan cuatro modelos clásicos de aprendizaje
automático para establecer la cota máxima de utilidad sin privacidad.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_bin, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bin
)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


def build_baseline_models():
    return {
        "Regresión Logística":    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
        "Random Forest":          RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, class_weight="balanced"),
        "Árbol de Decisión":      DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE, class_weight="balanced"),
        "Naive Bayes (Gaussian)": GaussianNB(),
    }


baseline_rows = []
for name, model in build_baseline_models().items():
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    baseline_rows.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions, average="weighted"),
        "k": 0,
    })

df_baseline = pd.DataFrame(baseline_rows).sort_values("F1-Score", ascending=False)
display(df_baseline.round(4))

## 6. Exportación para ARX Desktop

Para la fase de $k$-anonimidad se exportan los conjuntos de entrenamiento
y prueba al formato esperado por ARX (separador `;`) junto con las
cinco jerarquías de generalización descritas en la memoria.

In [ ]:
train_index, test_index = X_train.index, X_test.index
df_train_raw = df_reduced.loc[train_index].copy()
df_test_raw = df_reduced.loc[test_index].copy()

for column in ("admission_type_id", "time_in_hospital"):
    df_train_raw[column] = df_train_raw[column].astype(str)
    df_test_raw[column] = df_test_raw[column].astype(str)

df_train_raw.to_csv(INPUTS_DIR / "arx_train.csv", sep=";", index=False)
df_test_raw.to_csv(INPUTS_DIR / "arx_test.csv", sep=";", index=False)

In [ ]:
# HIERARCHIES_DIR ya existe (creado en la celda de configuración)

hierarchies = {
    "race.csv": """Caucasian;Caucasian;White;*
AfricanAmerican;AfricanAmerican;NonWhite;*
Hispanic;Hispanic;NonWhite;*
Asian;Asian;NonWhite;*
Other;Other;NonWhite;*""",
    "gender.csv": """Female;*
Male;*
Unknown/Invalid;*""",
    "age.csv": "\n".join([
        "[0-10);[0-20);[0-40);*", "[10-20);[0-20);[0-40);*",
        "[20-30);[20-40);[0-40);*", "[30-40);[20-40);[0-40);*",
        "[40-50);[40-60);[40-80);*", "[50-60);[40-60);[40-80);*",
        "[60-70);[60-80);[40-80);*", "[70-80);[60-80);[40-80);*",
        "[80-90);[80-100);[80-100);*", "[90-100);[80-100);[80-100);*",
    ]),
    "admission_type_id.csv": """1;Urgente;Conocido;*
2;Urgente;Conocido;*
3;Electivo;Conocido;*
4;Newborn;Conocido;*
5;Desconocido;Desconocido;*
6;Desconocido;Desconocido;*
7;Urgente;Conocido;*
8;Desconocido;Desconocido;*""",
}

time_lines = []
for i in range(1, 15):
    if i in (1, 2):       l1, l2 = "[1-2]", "[1-7]"
    elif i in (3, 4):     l1, l2 = "[3-4]", "[1-7]"
    elif i in (5, 6, 7):  l1, l2 = "[5-7]", "[1-7]"
    elif i in (8, 9, 10): l1, l2 = "[8-10]", "[8-14]"
    else:                 l1, l2 = "[11-14]", "[8-14]"
    time_lines.append(f"{i};{l1};{l2};*")
hierarchies["time_in_hospital.csv"] = "\n".join(time_lines)

for filename, content in hierarchies.items():
    (HIERARCHIES_DIR / filename).write_text(content)

La anonimización propiamente dicha se ejecuta de forma manual en ARX
Desktop. El procedimiento detallado figura en `docs/guia_arx.md`. Una vez
exportados los cinco archivos `arx_output_k<k>.csv`, se reanuda la
ejecución del cuaderno.

## 7. Evaluación post-ARX

In [ ]:
def evaluate_kanon(filepath, k_value):
    df_anon = pd.read_csv(filepath, sep=";")
    n_initial = len(df_anon)
    mask_suppressed = df_anon["race"].astype(str).str.fullmatch(r"\*")
    df_anon = df_anon[~mask_suppressed].copy()
    suppression_pct = (n_initial - len(df_anon)) / n_initial * 100

    y_anon = (df_anon["readmitted"] != "NO").astype(int)
    X_anon = df_anon.drop(columns=["readmitted"])
    X_test_raw_view = df_test_raw.drop(columns=["readmitted"])

    n_anon = len(X_anon)
    stacked = pd.concat([X_anon, X_test_raw_view], ignore_index=True)
    stacked_encoded = pd.get_dummies(stacked, drop_first=True)
    X_anon_array = stacked_encoded.iloc[:n_anon]
    X_test_array = stacked_encoded.iloc[n_anon:]

    sc = StandardScaler().fit(X_anon_array)
    X_anon_scaled = sc.transform(X_anon_array)
    X_test_scaled_local = sc.transform(X_test_array)

    y_test_local = (df_test_raw["readmitted"] != "NO").astype(int).values

    rows = []
    for name, model in build_baseline_models().items():
        model.fit(X_anon_scaled, y_anon)
        predictions = model.predict(X_test_scaled_local)
        rows.append({
            "Modelo": name,
            "Accuracy": accuracy_score(y_test_local, predictions),
            "F1-Score": f1_score(y_test_local, predictions, average="weighted"),
            "k": k_value,
            "pct_suprimido": round(suppression_pct, 2),
            "filas_efectivas": len(df_anon),
            "columnas_OHE": X_anon_array.shape[1],
        })
    return pd.DataFrame(rows)


results_kanon = [df_baseline.assign(
    pct_suprimido=0, filas_efectivas=len(X_train), columnas_OHE=X_encoded.shape[1]
)]
for k in K_VALUES:
    filepath = ARX_OUTPUTS_DIR / f"arx_output_k{k}.csv"
    if filepath.exists():
        results_kanon.append(evaluate_kanon(filepath, k))

df_kanon = pd.concat(results_kanon, ignore_index=True)
df_kanon.to_csv(RESULTS_KANON_DIR / "resultados_kanon.csv", index=False)
display(df_kanon.pivot(index="Modelo", columns="k", values="Accuracy").round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name in df_kanon["Modelo"].unique():
    subset = df_kanon[df_kanon["Modelo"] == model_name].sort_values("k")
    axes[0].plot(subset["k"], subset["Accuracy"], marker="o", label=model_name)
    axes[1].plot(subset["k"], subset["F1-Score"], marker="s", label=model_name)

for ax, ylabel, title in [
    (axes[0], "Accuracy", "Degradación de Accuracy con k"),
    (axes[1], "F1-Score", "Degradación de F1-Score con k"),
]:
    ax.set_xlabel("k"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xscale("symlog")
    ax.set_xticks([0] + K_VALUES); ax.set_xticklabels(["0"] + [str(k) for k in K_VALUES])
    ax.grid(alpha=0.3); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparativa_kanon.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Aplicación de Privacidad Diferencial

Se ejecuta el sweep $\\epsilon \\in \\{0,1\\ 0,5\\ 1\\ 5\\ 10\\}$ con veinte repeticiones (semillas dispersas) por
punto sobre la Regresión Logística y el Naive Bayes Gaussian. La cota
de sensibilidad $\\texttt{data\\_norm}$ se fija al percentil 95 de las normas $L_2$
del conjunto de entrenamiento escalado para evitar la sobrecalibración
del ruido por filas atípicas.

In [ ]:
data_norm = float(np.percentile(np.linalg.norm(X_train_scaled, axis=1), 95))
mins = np.percentile(X_train_scaled, 1, axis=0)
maxs = np.percentile(X_train_scaled, 99, axis=0)
print(f"data_norm (p95): {data_norm:.4f}")

dp_rows = []
for epsilon in EPSILON_VALUES:
    for rep in range(N_REPETITIONS_DP):
        seed = DP_SEEDS[rep]
        lr_dp = dpm.LogisticRegression(epsilon=epsilon, data_norm=data_norm, max_iter=100, random_state=seed)
        lr_dp.fit(X_train_scaled, y_train)
        predictions = lr_dp.predict(X_test_scaled)
        dp_rows.append({
            "modelo": "Regresión Logística", "epsilon": epsilon, "rep": rep,
            "Accuracy": accuracy_score(y_test, predictions),
            "F1-Score": f1_score(y_test, predictions, average="weighted"),
        })

        nb_dp = dpm.GaussianNB(epsilon=epsilon, bounds=(mins, maxs), random_state=seed)
        nb_dp.fit(X_train_scaled, y_train)
        predictions = nb_dp.predict(X_test_scaled)
        dp_rows.append({
            "modelo": "Naive Bayes (Gaussian)", "epsilon": epsilon, "rep": rep,
            "Accuracy": accuracy_score(y_test, predictions),
            "F1-Score": f1_score(y_test, predictions, average="weighted"),
        })

df_dp = pd.DataFrame(dp_rows)
df_dp.to_csv(RESULTS_DP_DIR / "resultados_dp.csv", index=False)

dp_aggregated = df_dp.groupby(["modelo", "epsilon"]).agg(
    Acc_mean=("Accuracy", "mean"), Acc_std=("Accuracy", "std"),
    F1_mean=("F1-Score", "mean"), F1_std=("F1-Score", "std"),
).round(4)
display(dp_aggregated)

## 9. Auditoría adversarial: MIA Black-Box

In [ ]:
def run_mia(model, X_train_array, y_train_array, X_test_array, y_test_array, label):
    art_classifier = ScikitlearnLogisticRegression(model=model)
    attack = MembershipInferenceBlackBox(estimator=art_classifier, attack_model_type="rf")
    rng = np.random.default_rng(RANDOM_STATE)

    n_tr = min(len(X_train_array), 10000); n_te = min(len(X_test_array), 5000)
    idx_tr = rng.choice(len(X_train_array), n_tr, replace=False)
    idx_te = rng.choice(len(X_test_array), n_te, replace=False)

    Xa = X_train_array[idx_tr]; ya = np.asarray(y_train_array)[idx_tr]
    Xt = X_test_array[idx_te];  yt = np.asarray(y_test_array)[idx_te]
    half_tr = n_tr // 2; half_te = n_te // 2

    attack.fit(x=Xa[:half_tr], y=ya[:half_tr], test_x=Xt[:half_te], test_y=yt[:half_te])
    inferred_members = attack.infer(Xa[half_tr:], ya[half_tr:])
    inferred_non = attack.infer(Xt[half_te:], yt[half_te:])

    attack_acc = ((inferred_members == 1).sum() + (inferred_non == 0).sum()) / (len(inferred_members) + len(inferred_non))
    proba_m_ = np.asarray(attack.infer(Xa[half_tr:], ya[half_tr:], probabilities=True)).reshape(-1)
    proba_n_ = np.asarray(attack.infer(Xt[half_te:], yt[half_te:], probabilities=True)).reshape(-1)
    if len(proba_m_) == 2 * (n_tr - half_tr):
        proba_m_ = proba_m_.reshape(n_tr - half_tr, 2)[:, -1]
        proba_n_ = proba_n_.reshape(n_te - half_te, 2)[:, -1]
    _scores = np.concatenate([proba_m_, proba_n_])
    _labels = np.concatenate([np.ones(len(proba_m_), dtype=int), np.zeros(len(proba_n_), dtype=int)])
    try:
        attack_auc = float(roc_auc_score(_labels, _scores))
    except ValueError:
        attack_auc = float('nan')
    advantage = float((inferred_members == 1).mean() - (inferred_non == 1).mean())
    utility = float(accuracy_score(y_test_array, model.predict(X_test_array)))
    return {"escenario": label, "utility_acc": round(utility, 4),
            "attack_acc": round(float(attack_acc), 4), "attack_advantage": round(advantage, 4)}


mia_rows = []
lr_baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced").fit(X_train_scaled, y_train)
mia_rows.append(run_mia(lr_baseline, X_train_scaled, y_train.values, X_test_scaled, y_test.values, "Baseline LR"))

for epsilon in (0.1, 1.0, 10.0):
    lr_dp = dpm.LogisticRegression(epsilon=epsilon, data_norm=data_norm, max_iter=100, random_state=RANDOM_STATE)
    lr_dp.fit(X_train_scaled, y_train)
    mia_rows.append(run_mia(lr_dp, X_train_scaled, y_train.values, X_test_scaled, y_test.values, f"LR DP epsilon={epsilon}"))

df_mia = pd.DataFrame(mia_rows)
df_mia.to_csv(RESULTS_MIA_DIR / "mia_results.csv", index=False)
display(df_mia)

## 10. Tests de significancia estadística

Los tests de McNemar contrastan las predicciones del baseline frente a
cada modelo $k$-anonimizado. Los tests de Wilcoxon de una muestra
contrastan la Accuracy promedio de los modelos diferencialmente privados
frente al baseline sin privacidad.

In [ ]:
def mcnemar_test(y_true, predictions_a, predictions_b):
    correct_a = (predictions_a == y_true); correct_b = (predictions_b == y_true)
    b = int(((correct_a) & (~correct_b)).sum()); c = int(((~correct_a) & (correct_b)).sum())
    if b + c == 0:
        return 1.0, 0, 0, 0.0
    chi_squared = (abs(b - c) - 1) ** 2 / (b + c)
    return float(1 - chi2.cdf(chi_squared, df=1)), b, c, float(chi_squared)


baseline_predictions = {}
for name, model in build_baseline_models().items():
    model.fit(X_train_scaled, y_train)
    baseline_predictions[name] = model.predict(X_test_scaled)

mcnemar_rows = []
for k in K_VALUES:
    filepath = ARX_OUTPUTS_DIR / f"arx_output_k{k}.csv"
    if not filepath.exists():
        continue
    df_k = pd.read_csv(filepath, sep=";")
    df_k = df_k[~df_k["race"].astype(str).str.fullmatch(r"\*")]
    y_k = (df_k["readmitted"] != "NO").astype(int).values
    X_k = df_k.drop(columns=["readmitted"])
    X_test_view = df_test_raw.drop(columns=["readmitted"])
    n_k = len(X_k)
    encoded = pd.get_dummies(pd.concat([X_k, X_test_view], ignore_index=True), drop_first=True)
    Xa_array = encoded.iloc[:n_k]; Xt_array = encoded.iloc[n_k:]
    sc_local = StandardScaler().fit(Xa_array)
    for name, model in build_baseline_models().items():
        model.fit(sc_local.transform(Xa_array), y_k)
        predictions_k = model.predict(sc_local.transform(Xt_array))
        p_value, b, c, chi_sq = mcnemar_test(y_test.values, baseline_predictions[name], predictions_k)
        mcnemar_rows.append({
            "modelo": name, "k": k, "b": b, "c": c,
            "chi2": chi_sq, "p_value": p_value, "significativo_005": p_value < 0.05,
        })

df_mcnemar = pd.DataFrame(mcnemar_rows)
# Corrección Bonferroni para N=20 tests (5 k × 4 modelos)
_n_mc = len(df_mcnemar)
df_mcnemar["p_bonferroni"] = (df_mcnemar["p_value"] * _n_mc).clip(upper=1.0)
df_mcnemar["significativo_bonf"] = df_mcnemar["p_bonferroni"] < 0.05
df_mcnemar = apply_bonferroni(df_mcnemar)  # N=20 (4 modelos × 5 k)
df_mcnemar.to_csv(RESULTS_KANON_DIR / "mcnemar_kanon.csv", index=False)
display(df_mcnemar.round(4))

In [ ]:
baselines_global = {row["Modelo"]: round(float(row["Accuracy"]), 4) for _, row in df_kanon[df_kanon["k"]==0].iterrows()}  # leído de resultados_kanon.csv
wilcoxon_rows = []
for model_name, baseline_acc in baselines_global.items():
    for epsilon in EPSILON_VALUES:
        sample = df_dp[(df_dp.modelo == model_name) & (df_dp.epsilon == epsilon)]["Accuracy"].values
        try:
            statistic, p_value = wilcoxon(sample - baseline_acc)
        except ValueError:
            p_value = 1.0
        wilcoxon_rows.append({
            "modelo": model_name, "epsilon": epsilon,
            "mean_acc": float(np.mean(sample)), "baseline": baseline_acc,
            "p_value": float(p_value),
            "delta_pct": (float(np.mean(sample)) - baseline_acc) / baseline_acc * 100,
        })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
# Corrección Bonferroni para N=10 tests (5 ε × 2 modelos)
_n_w = len(df_wilcoxon)
df_wilcoxon["p_bonferroni"] = (df_wilcoxon["p_value"] * _n_w).clip(upper=1.0)
df_wilcoxon["significativo_bonf"] = df_wilcoxon["p_bonferroni"] < 0.05
df_wilcoxon = apply_bonferroni(df_wilcoxon)  # N=10 (2 modelos × 5 ε)
df_wilcoxon.to_csv(RESULTS_DP_DIR / "wilcoxon_dp.csv", index=False)
display(df_wilcoxon.round(4))

## 11. Defensa en profundidad: combinación $k=10$ + DP

Replica el sweep $\epsilon \in \{0{,}1, 1, 10\}$ con $n=20$ semillas
dispersas sobre el conjunto previamente $k=10$-anonimizado. Mide si la
composición compensa o agrava el efecto de la DP sobre los grupos
minoritarios (Sección 4.5 de la memoria). Llama a `dp_on_kanonimized`
de `src.differential_privacy` para garantizar identidad con `scripts/05`.

In [ ]:
filepath_k10 = ARX_OUTPUTS_DIR / "arx_output_k10.csv"
assert filepath_k10.exists(), "Ejecuta ARX para k=10 antes de esta fase."

df_combo = dp_on_kanonimized(
    filepath_kanon=filepath_k10,
    df_test_raw=df_test_raw,
)
df_combo.to_csv(RESULTS_DP_DIR / "resultados_combo.csv", index=False)

aggregated = df_combo.groupby(["modelo", "epsilon"])["Accuracy"].agg(["mean", "std"]).round(4)
print("Resultados combinación k=10 + DP (Accuracy mean ± std, n=20):")
print(aggregated.to_string())

## 12. Extensión sintáctica: $l$-diversidad y $t$-\textit{closeness}

Evalúa los nueve conjuntos anonimizados exportados desde ARX con
`diag_1_category` marcado como Sensitive. Para cada configuración
entrena los cuatro modelos baseline, mide utilidad sobre el test set,
equidad por subgrupos `race` y verifica las restricciones de privacidad
declaradas. Idéntico a `scripts/08_eval_ldiv_tclos.py`.

In [ ]:
CONFIGURATIONS_LT = [
    ("arx_output_k5.csv",          5, None, None, "k_solo"),
    ("arx_output_k5_l2.csv",       5, 2,    None, "l_div"),
    ("arx_output_k5_l3.csv",       5, 3,    None, "l_div"),
    ("arx_output_k5_l5.csv",       5, 5,    None, "l_div"),
    ("arx_output_k5_t05.csv",      5, None, 0.5,  "t_clos"),
    ("arx_output_k5_t04.csv",      5, None, 0.4,  "t_clos"),
    ("arx_output_k5_t035.csv",     5, None, 0.35, "t_clos"),
    ("arx_output_k5_t03.csv",      5, None, 0.3,  "t_clos"),
    ("arx_output_k5_t025.csv",     5, None, 0.25, "t_clos"),
    ("arx_output_k5_l2_t05.csv",   5, 2,    0.5,  "triple_soft"),
    ("arx_output_k5_l3_t035.csv",  5, 3,    0.35, "triple_medium"),
    ("arx_output_k5_l5_t03.csv",   5, 5,    0.3,  "triple_hard"),
    ("arx_output_k5_l5_t025.csv",  5, 5,    0.25, "triple_extreme"),
]

utility_parts = []
fairness_parts = [fairness_ldiv_tclos(
    None, df_train_raw, df_test_raw,
    {"k": 5, "l": None, "t": None, "tipo": "k_solo", "archivo": "baseline"},
)]
verifications = []

for filename, k, l, t, tipo in CONFIGURATIONS_LT:
    filepath = ARX_OUTPUTS_DIR / filename
    if not filepath.exists():
        print(f"  AVISO: falta {filename}, se omite")
        continue
    tag = {"k": k, "l": l, "t": t, "tipo": tipo, "archivo": filename}
    utility_parts.append(evaluate_ldiv_tclos(filepath, df_test_raw, tag))
    fairness_parts.append(fairness_ldiv_tclos(filepath, df_train_raw, df_test_raw, tag))
    verifications.append(verify_constraints(
        filepath, _cfg.QID_COLUMNS, df_train_raw,
        expected_k=k, expected_l=l, expected_t=t,
    ))
    print(f"  ✓ {filename}")

df_lt_utility = pd.concat(utility_parts, ignore_index=True)
df_lt_utility.to_csv(RESULTS_LDIV_TCLOS_DIR / "resultados_ldiv_tclos.csv", index=False)
df_lt_fairness = pd.concat(fairness_parts, ignore_index=True)
df_lt_fairness.to_csv(RESULTS_LDIV_TCLOS_DIR / "fairness_ldiv_tclos.csv", index=False)
pd.DataFrame(verifications).to_csv(
    RESULTS_LDIV_TCLOS_DIR / "verificacion_ldiv_tclos.csv", index=False
)
print(f"\n✓ resultados_ldiv_tclos.csv ({len(df_lt_utility)} filas)")
print(f"✓ fairness_ldiv_tclos.csv ({len(df_lt_fairness)} filas)")
print(f"✓ verificacion_ldiv_tclos.csv ({len(verifications)} filas)")

## 13. MIA Black-Box sobre las configuraciones estrictas $l$/$t$

Audita Regresión Logística sobre cuatro escenarios extremos
($k=5$ referencia, $k=5+l=5$, $k=5+t=0{,}25$ y triple Hard) con
muestreo reducido ($n_{\text{auditor}}=2{.}000$) y semillas
independientes por configuración. Equivalente a `scripts/09_mia_ldiv_tclos.py`.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.estimators.classification.scikitlearn import ScikitlearnLogisticRegression

def _arrays_from_anon_lt(filepath, df_test_raw):
    df_anon = pd.read_csv(filepath, sep=";")
    df_anon = df_anon[~df_anon["race"].astype(str).str.fullmatch(r"\*")]
    y_anon = binarize_target(df_anon[_cfg.TARGET_COLUMN]).values
    X_anon = df_anon.drop(columns=[_cfg.TARGET_COLUMN])
    X_test_local = df_test_raw.drop(columns=[_cfg.TARGET_COLUMN])
    Xa, Xt = joint_ohe(X_anon, X_test_local)
    sc = StandardScaler().fit(Xa)
    return sc.transform(Xa).astype(np.float32), y_anon, sc.transform(Xt).astype(np.float32)

def _mia_attack_lt(X_train, y_train, X_test, y_test, seed, n_attack=2000):
    target = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    target.fit(X_train, y_train)
    art_classifier = ScikitlearnLogisticRegression(model=target, clip_values=(X_train.min(), X_train.max()))
    rng = np.random.default_rng(seed)
    n_attack = min(n_attack, len(X_test))
    attack_idx = rng.choice(len(X_train), n_attack, replace=False)
    test_idx = rng.choice(len(X_test), n_attack, replace=False)
    attack = MembershipInferenceBlackBox(estimator=art_classifier, attack_model_type="rf")
    half = n_attack // 2
    attack.fit(X_train[attack_idx[:half]], y_train[attack_idx[:half]], X_test[test_idx[:half]], y_test[test_idx[:half]])
    inf_m = attack.infer(X_train[attack_idx[half:]], y_train[attack_idx[half:]])
    inf_n = attack.infer(X_test[test_idx[half:]], y_test[test_idx[half:]])
    proba_m = np.asarray(attack.infer(X_train[attack_idx[half:]], y_train[attack_idx[half:]], probabilities=True)).reshape(-1)
    proba_n = np.asarray(attack.infer(X_test[test_idx[half:]], y_test[test_idx[half:]], probabilities=True)).reshape(-1)
    if len(proba_m) == 2 * half:
        proba_m = proba_m.reshape(half, 2)[:, -1]
        proba_n = proba_n.reshape(half, 2)[:, -1]
    scores = np.concatenate([proba_m, proba_n])
    labels = np.concatenate([np.ones(half, dtype=int), np.zeros(half, dtype=int)])
    try: auc = float(roc_auc_score(labels, scores))
    except ValueError: auc = float("nan")
    tpr, fpr = float(inf_m.mean()), float(inf_n.mean())
    return tpr, fpr, tpr - fpr, auc

targets_lt = [
    ("k=5 (referencia)",          "arx_output_k5.csv"),
    ("k=5 + l=5",                 "arx_output_k5_l5.csv"),
    ("k=5 + t=0.25",              "arx_output_k5_t025.csv"),
    ("k=5 + l=5 + t=0.3 (Hard)",  "arx_output_k5_l5_t03.csv"),
]

mia_lt_rows = []
for i, (label, filename) in enumerate(targets_lt):
    filepath = ARX_OUTPUTS_DIR / filename
    if not filepath.exists():
        print(f"  AVISO: falta {filename}"); continue
    Xa, ya, Xt = _arrays_from_anon_lt(filepath, df_test_raw)
    seed = DP_SEEDS[i % len(DP_SEEDS)]
    tpr, fpr, adv, auc = _mia_attack_lt(Xa, ya, Xt, y_test.values, seed=seed)
    mia_lt_rows.append({
        "escenario": label, "archivo": filename, "seed": seed,
        "TPR": round(tpr, 4), "FPR": round(fpr, 4),
        "Advantage": round(adv, 4), "AUC": round(auc, 4),
    })
    print(f"  ✓ {label} (seed={seed})")

df_mia_lt = pd.DataFrame(mia_lt_rows)
df_mia_lt.to_csv(RESULTS_MIA_DIR / "mia_ldiv_tclos.csv", index=False)
print(df_mia_lt.round(4).to_string(index=False))

## 14. Test de McNemar sobre las configuraciones extremas $l$/$t$

Replica la batería de McNemar de la §10 sobre cuatro configuraciones
estrictas de la extensión sintáctica con corrección de Bonferroni
($N=16$, $\alpha_{\text{efectivo}} = 0{,}0031$). Equivalente a
`scripts/11_mcnemar_ldiv_tclos.py`.

In [ ]:
CFG_MCN_LT = [
    ("k=5 + l=5",                "arx_output_k5_l5.csv"),
    ("k=5 + t=0.5",              "arx_output_k5_t05.csv"),
    ("k=5 + t=0.25",             "arx_output_k5_t025.csv"),
    ("k=5 + l=5 + t=0.3 (Hard)", "arx_output_k5_l5_t03.csv"),
]

# Las predicciones del baseline ya están en `baseline_predictions` (cell 24)
mcn_lt_rows = []
for label, filename in CFG_MCN_LT:
    filepath = ARX_OUTPUTS_DIR / filename
    if not filepath.exists():
        print(f"  AVISO: falta {filename}"); continue
    df_anon = pd.read_csv(filepath, sep=";")
    df_anon = df_anon[~df_anon["race"].astype(str).str.fullmatch(r"\*")].copy()
    y_anon = binarize_target(df_anon[_cfg.TARGET_COLUMN]).values
    X_anon = df_anon.drop(columns=[_cfg.TARGET_COLUMN])
    X_test_lt = df_test_raw.drop(columns=[_cfg.TARGET_COLUMN])
    Xa, Xt = joint_ohe(X_anon, X_test_lt)
    sc_lt = StandardScaler().fit(Xa)
    for name, model in build_baseline_models().items():
        model.fit(sc_lt.transform(Xa), y_anon)
        pred = model.predict(sc_lt.transform(Xt))
        p, b, c, chi2_v = mcnemar_test(y_test.values, baseline_predictions[name], pred)
        mcn_lt_rows.append({
            "configuracion": label, "archivo": filename, "modelo": name,
            "b": b, "c": c, "chi2": round(chi2_v, 4),
            "p_value": p, "significativo_005": p < 0.05,
        })
    print(f"  ✓ {label}")

df_mcn_lt = apply_bonferroni(pd.DataFrame(mcn_lt_rows))  # N=16 (4 cfg × 4 modelos)
df_mcn_lt.to_csv(RESULTS_LDIV_TCLOS_DIR / "mcnemar_ldiv_tclos.csv", index=False)
print(df_mcn_lt.round(4).to_string(index=False))

## 15. Artefactos generados

El cuaderno produce las tablas y figuras consumidas por la memoria:

| Archivo | Descripción |
|---|---|
| `resultados_kanon.csv` | Métricas por modelo y nivel de $k$ |
| `resultados_dp.csv`    | 200 ejecuciones del sweep ε × 20 repeticiones (semillas dispersas) |
| `mcnemar_kanon.csv`    | Tests de significancia ARX |
| `wilcoxon_dp.csv`      | Tests de significancia DP |
| `mia_results.csv`      | Auditoría adversarial Black-Box |
| `comparativa_kanon.png` | Curvas de degradación ARX |